# Notebook 3 — IndiaFinBench: Model Evaluation & Error Analysis

Reproduces all tables and figures from Section 5–6 of the paper.
Run from `IndiaFinBench/`.

In [ ]:
import csv
import math
import re
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

ROOT        = Path('..')
RESULTS_DIR = ROOT / 'evaluation/results'

MODEL_ORDER  = ['haiku', 'gemini', 'groq70b', 'llama3', 'mistral']
MODEL_LABELS = {
    'haiku':   'Claude 3 Haiku',
    'gemini':  'Gemini 2.5 Flash',
    'groq70b': 'LLaMA-3.3-70B',
    'llama3':  'LLaMA-3-8B',
    'mistral': 'Mistral-7B',
}
TASK_ORDER = ['regulatory_interpretation','numerical_reasoning',
              'contradiction_detection','temporal_reasoning']
TASK_SHORT = {'regulatory_interpretation':'REG','numerical_reasoning':'NUM',
              'contradiction_detection':'CON','temporal_reasoning':'TMP'}

def load_results(mk):
    path = RESULTS_DIR / f'{mk}_results.csv'
    rows = []
    with path.open(encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if 'FAIL' not in row.get('prediction',''):
                rows.append(row)
    return rows

all_results = {mk: load_results(mk) for mk in MODEL_ORDER}
for mk, rows in all_results.items():
    print(f'{MODEL_LABELS[mk]:<22}: {len(rows)} valid rows')

In [ ]:
# Cell 2 — Reproduce Table 5 (main results) as styled pandas DataFrame

def task_acc(rows, task):
    t = [r for r in rows if r['task_type'] == task]
    return sum(int(r['correct']) for r in t) / len(t) * 100 if t else 0.0

def overall_acc(rows):
    return sum(int(r['correct']) for r in rows) / len(rows) * 100 if rows else 0.0

table5_data = []
for mk in MODEL_ORDER:
    rows = all_results[mk]
    row  = {'Model': MODEL_LABELS[mk]}
    for task in TASK_ORDER:
        row[TASK_SHORT[task]] = round(task_acc(rows, task), 1)
    row['Overall'] = round(overall_acc(rows), 1)
    table5_data.append(row)

table5 = pd.DataFrame(table5_data).set_index('Model')

def highlight_best(s):
    """Green = best, red = worst per column."""
    is_best  = s == s.max()
    is_worst = s == s.min()
    return ['background-color: #c6efce; font-weight: bold' if b
            else ('background-color: #ffc7ce' if w else '')
            for b, w in zip(is_best, is_worst)]

styled = table5.style.apply(highlight_best).format('{:.1f}%')
print('Table 5 — Main Results (accuracy %)')
print(table5.to_string())
styled

In [ ]:
# Cell 3 — Reproduce Figure 1 (heatmap) inline

matrix = np.array([[table5.loc[MODEL_LABELS[mk], TASK_SHORT[t]]
                    for t in TASK_ORDER] for mk in MODEL_ORDER])

task_labels  = [t.replace('_', '\n') for t in TASK_ORDER]
model_labels = [MODEL_LABELS[mk] for mk in MODEL_ORDER]

cmap = plt.cm.RdYlGn
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(matrix, cmap=cmap, vmin=60, vmax=100, aspect='auto')

ax.set_xticks(range(len(TASK_ORDER)))
ax.set_yticks(range(len(MODEL_ORDER)))
ax.set_xticklabels(task_labels, fontsize=10)
ax.set_yticklabels(model_labels, fontsize=10)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

for i in range(len(MODEL_ORDER)):
    for j in range(len(TASK_ORDER)):
        val = matrix[i, j]
        norm_val = (val - 60) / 40.0
        rgba = cmap(np.clip(norm_val, 0, 1))
        lum  = 0.299*rgba[0] + 0.587*rgba[1] + 0.114*rgba[2]
        color = 'black' if lum > 0.45 else 'white'
        ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                fontsize=11, fontweight='bold', color=color)

plt.colorbar(im, ax=ax, shrink=0.8).set_label('Accuracy (%)', fontsize=10)
ax.set_title('Figure 1 — IndiaFinBench: Model Performance by Task Type',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 — Reproduce Figure 2 (difficulty breakdown) inline

difficulties = ['easy', 'medium', 'hard']
colors_diff  = ['#4CAF50', '#FF9800', '#F44336']

diff_data = {}
for mk in MODEL_ORDER:
    rows = all_results[mk]
    diff_data[mk] = {}
    for d in difficulties:
        d_rows = [r for r in rows if r.get('difficulty','') == d]
        diff_data[mk][d] = sum(int(r['correct']) for r in d_rows)/len(d_rows)*100 if d_rows else 0

x     = np.arange(len(MODEL_ORDER))
width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))

for i, (d, c) in enumerate(zip(difficulties, colors_diff)):
    vals = [diff_data[mk][d] for mk in MODEL_ORDER]
    bars = ax.bar(x + i*width, vals, width, label=d.capitalize(),
                  color=c, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val:.0f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels([MODEL_LABELS[mk] for mk in MODEL_ORDER], fontsize=9)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 115)
ax.set_title('Figure 2 — Accuracy by Question Difficulty', fontsize=13, fontweight='bold')
ax.legend(title='Difficulty')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — Reproduce Figure 3 (error stacked bar) inline

error_data = {
    'haiku':   {'DKF': 4,  'NRF': 2,  'TRF': 7,  'CGF': 0},
    'gemini':  {'DKF': 3,  'NRF': 5,  'TRF': 9,  'CGF': 1},
    'groq70b': {'DKF': 12, 'NRF': 5,  'TRF': 10, 'CGF': 1},
    'llama3':  {'DKF': 13, 'NRF': 12, 'TRF': 11, 'CGF': 1},
    'mistral': {'DKF': 17, 'NRF': 10, 'TRF': 13, 'CGF': 1},
}
error_types  = ['DKF', 'NRF', 'TRF', 'CGF']
error_colors = ['#4472C4', '#ED7D31', '#A9D18E', '#FF0000']
error_labels = {'DKF': 'Domain Knowledge Failure', 'NRF': 'Numerical Reasoning Failure',
                'TRF': 'Temporal Reasoning Failure', 'CGF': 'Context Grounding Failure'}

fig, ax = plt.subplots(figsize=(11, 5))
y_pos  = np.arange(len(MODEL_ORDER))
lefts  = np.zeros(len(MODEL_ORDER))

for et, color in zip(error_types, error_colors):
    totals = [sum(error_data[mk].values()) or 1 for mk in MODEL_ORDER]
    widths = np.array([error_data[mk][et]/tot*100 for mk, tot in zip(MODEL_ORDER, totals)])
    bars = ax.barh(y_pos, widths, left=lefts, color=color,
                   label=error_labels[et], edgecolor='white', linewidth=0.5)
    for bar, w, left in zip(bars, widths, lefts):
        if w > 8:
            ax.text(left+w/2, bar.get_y()+bar.get_height()/2,
                    f'{w:.0f}%', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')
    lefts += widths

ax.set_yticks(y_pos)
ax.set_yticklabels([MODEL_LABELS[mk] for mk in MODEL_ORDER], fontsize=10)
ax.set_xlabel('Percentage of Total Errors (%)')
ax.set_xlim(0, 100)
ax.set_title('Figure 3 — Error Type Distribution by Model', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=8, framealpha=0.9)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Per-task analysis: which task has highest variance across models?

task_accs = {t: [] for t in TASK_ORDER}
for mk in MODEL_ORDER:
    rows = all_results[mk]
    for task in TASK_ORDER:
        task_accs[task].append(task_acc(rows, task))

print('Per-task accuracy variance across models:')
print(f'  {"Task":<32}  {"Min":>6}  {"Max":>6}  {"Mean":>6}  {"Spread":>7}  {"Std":>6}')
print(f'  {"-"*32}  {"-"*6}  {"-"*6}  {"-"*6}  {"-"*7}  {"-"*6}')
variances = {}
for task in TASK_ORDER:
    accs   = task_accs[task]
    spread = max(accs) - min(accs)
    variances[task] = np.var(accs)
    print(f'  {task:<32}  {min(accs):>5.1f}%  {max(accs):>5.1f}%  '
          f'{np.mean(accs):>5.1f}%  {spread:>6.1f}pp  {np.std(accs):>5.1f}')

most_var = max(variances, key=variances.get)
print(f'\nHighest variance task: {most_var} (var={variances[most_var]:.1f})')

In [ ]:
# Cell 7 — Confusion analysis for contradiction detection: Yes vs No per model

def extract_yn(text):
    t = str(text).lower().strip()
    if t.startswith('yes'): return 'Yes'
    if t.startswith('no'):  return 'No'
    return 'Other'

print('Contradiction detection — prediction distribution (Yes / No / Other):')
print(f'  {"Model":<22}  {"Yes":>6}  {"No":>6}  {"Other":>6}  {"Acc":>6}')
print(f'  {"-"*22}  {"-"*6}  {"-"*6}  {"-"*6}  {"-"*6}')

for mk in MODEL_ORDER:
    rows = [r for r in all_results[mk] if r['task_type'] == 'contradiction_detection']
    preds = [extract_yn(r['prediction']) for r in rows]
    n_yes   = preds.count('Yes')
    n_no    = preds.count('No')
    n_other = preds.count('Other')
    n_tot   = len(rows)
    acc     = sum(int(r['correct']) for r in rows)/n_tot*100 if n_tot else 0
    print(f'  {MODEL_LABELS[mk]:<22}  {n_yes:>6}  {n_no:>6}  {n_other:>6}  {acc:>5.1f}%')

# Reference Yes/No distribution
all_con_rows = [r for rows in all_results.values()
                for r in rows if r['task_type'] == 'contradiction_detection']
refs = [extract_yn(r['ref_answer']) for r in all_con_rows[:30]]  # use one model's set
print(f'\nReference answer distribution (n=30): Yes={refs.count("Yes")}, No={refs.count("No")}')

In [ ]:
# Cell 8 — Wilson CI for Numerical Reasoning (most discriminative task)

Z95 = 1.96

def wilson_ci(n_correct, n, z=Z95):
    if n == 0: return 0.0, 0.0, 0.0
    p   = n_correct / n
    z2n = z*z / n
    ctr = (p + z2n/2) / (1 + z2n)
    hw  = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / (1 + z2n)
    return max(0, ctr-hw)*100, min(1, ctr+hw)*100, hw*100

print('Numerical Reasoning — Wilson 95% CI (most discriminative task):')
print(f'  {"Model":<22}  {"n":>4}  {"Correct":>7}  {"Acc":>6}  {"95% CI":>18}  {"HW":>6}')
print(f'  {"-"*22}  {"-"*4}  {"-"*7}  {"-"*6}  {"-"*18}  {"-"*6}')
for mk in MODEL_ORDER:
    rows  = [r for r in all_results[mk] if r['task_type'] == 'numerical_reasoning']
    n     = len(rows)
    ncorr = sum(int(r['correct']) for r in rows)
    lo, hi, hw = wilson_ci(ncorr, n)
    acc   = ncorr/n*100 if n else 0
    print(f'  {MODEL_LABELS[mk]:<22}  {n:>4}  {ncorr:>7}  {acc:>5.1f}%  '
          f'[{lo:>5.1f}%, {hi:>5.1f}%]  {hw:>5.1f}pp')

In [ ]:
# Cell 9 — Bootstrap p-values for top 3 model comparisons
np.random.seed(42)

def load_aligned(mk_a, mk_b):
    rows_a = {r['id']: r for r in all_results[mk_a]}
    rows_b = {r['id']: r for r in all_results[mk_b]}
    shared = sorted(set(rows_a) & set(rows_b))
    sa = np.array([int(rows_a[i]['correct']) for i in shared])
    sb = np.array([int(rows_b[i]['correct']) for i in shared])
    return sa, sb

def paired_bootstrap(sa, sb, n=10_000):
    obs   = np.mean(sa) - np.mean(sb)
    delta = sa - sb
    cent  = delta - np.mean(delta)
    rng   = np.random.default_rng(42)
    boot  = cent[rng.integers(0, len(cent), (n, len(cent)))].mean(axis=1)
    return float(np.mean(np.abs(boot) >= np.abs(obs)))

focus_pairs = [
    ('haiku',  'gemini',  'Haiku vs Gemini'),
    ('haiku',  'groq70b', 'Haiku vs LLaMA-70B'),
    ('gemini', 'groq70b', 'Gemini vs LLaMA-70B'),
]

print('Bootstrap p-values (n=10,000 resamples, seed=42):')
print(f'  {"Comparison":<28}  {"Acc A":>6}  {"Acc B":>6}  {"Delta":>6}  {"p-val":>7}  Sig?')
print(f'  {"-"*28}  {"-"*6}  {"-"*6}  {"-"*6}  {"-"*7}  {"-"*5}')
for mk_a, mk_b, label in focus_pairs:
    sa, sb  = load_aligned(mk_a, mk_b)
    p       = paired_bootstrap(sa, sb)
    acc_a   = np.mean(sa)*100
    acc_b   = np.mean(sb)*100
    delta   = acc_a - acc_b
    sig     = 'YES *' if p < 0.05 else 'no'
    print(f'  {label:<28}  {acc_a:>5.1f}%  {acc_b:>5.1f}%  {delta:>+5.1f}%  {p:>7.4f}  {sig}')

In [ ]:
# Cell 10 — Formatted leaderboard matching Table 5
print('\n' + '='*72)
print('  INDIAFINBENCH LEADERBOARD (Table 5)')
print('='*72)
print(f'  {"Model":<22}  {"REG":>6}  {"NUM":>6}  {"CON":>6}  {"TMP":>6}  {"Overall":>8}')
print(f'  {"-"*22}  {"-"*6}  {"-"*6}  {"-"*6}  {"-"*6}  {"-"*8}')

ranked = sorted(
    MODEL_ORDER,
    key=lambda mk: overall_acc(all_results[mk]),
    reverse=True
)

for mk in ranked:
    rows = all_results[mk]
    cols = []
    for task in TASK_ORDER:
        a = task_acc(rows, task)
        cols.append(f'{a:>5.1f}%')
    ov = overall_acc(rows)
    print(f'  {MODEL_LABELS[mk]:<22}  {"  ".join(cols)}  {ov:>7.1f}%')

print('='*72)